# R_final Benchmark Validation
Compare R(s) vs R_final on standard benchmarks (M4, DetectRL, RealDet, XSum).
R_final uses: CE variance + agreement rate + confidence-coherence.
Goal: R_final should maintain standard detection while being robust to humanization.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch==2.5.1+cu121', 'torchvision==0.20.1+cu121', '--index-url', 'https://download.pytorch.org/whl/cu121', '-q'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers==4.46.3', 'accelerate==1.1.1', 'bitsandbytes', 'scipy', 'scikit-learn', '-q'])
import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0); print(f'GPU: {p.name}, compute {p.major}.{p.minor}')
    x = torch.randn(2,2).cuda(); print(f'OK: {(x@x).sum().item():.2f}'); del x

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 97.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 72.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 108.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.5.1+cu121 which is incompatible.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 108.2 MB/s eta 0:00:00
PyTorch 2.5.1+cu121, CUDA: True
GPU: Tesla P100-PCIE-16GB, compute 6.0
OK: 1.07


In [2]:
import gc, os, sys, json, shutil, warnings
import numpy as np
import torch, torch.nn.functional as F
from tqdm import tqdm
from scipy import stats
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')

BASE = '/kaggle/input'
df = ds = None
for r, dirs, files in os.walk(BASE):
    for f in files:
        if f == 'benchmark_dataset.json': df = os.path.join(r, f)
        if f == '__init__.py' and 'dna_detectllm' in r: ds = os.path.dirname(os.path.join(r, f))
    for d in dirs:
        if d == 'dna_detectllm':
            c = os.path.join(r, d)
            if os.path.isfile(os.path.join(c, '__init__.py')): ds = c
if not os.path.exists('dna_detectllm') and ds: shutil.copytree(ds, 'dna_detectllm', dirs_exist_ok=True)
sys.path.insert(0, os.getcwd())
from dna_detectllm.metrics import sum_perplexity, entropy
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
with open(df) as f: dataset = json.load(f)
print(f'Benchmarks: {list(dataset["benchmarks"].keys())}')

Benchmarks: ['M4', 'DetectRL', 'RealDet', 'XSum_GPT4']


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import ctypes
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gc.collect(); torch.cuda.empty_cache()
tokenizer = AutoTokenizer.from_pretrained('tiiuae/falcon-7b')
tokenizer.pad_token = tokenizer.eos_token
MAX_LEN = 256; DEVICE = 'cuda:0'
cc = torch.cuda.get_device_properties(0)
if (cc.major, cc.minor) >= (7,0):
    from transformers import BitsAndBytesConfig
    qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    lk = dict(quantization_config=qc, device_map='auto', low_cpu_mem_usage=True); MAX_LEN = 512
else:
    lk = dict(torch_dtype=torch.float16, device_map='auto', low_cpu_mem_usage=True)
observer = AutoModelForCausalLM.from_pretrained('tiiuae/falcon-7b', **lk); observer.eval()
gc.collect(); torch.cuda.empty_cache()
try: ctypes.CDLL('libc.so.6').malloc_trim(0)
except: pass
performer = AutoModelForCausalLM.from_pretrained('tiiuae/falcon-7b-instruct', **lk); performer.eval()
print(f'Models loaded. MAX_LEN={MAX_LEN}')

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

2026-04-15 14:31:18.842832: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776263479.014801      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776263479.065492      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776263479.494749      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776263479.494785      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776263479.494788      23 computation_placer.cc:177] computation placer alr

...

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Models loaded. MAX_LEN=256


In [4]:
@torch.inference_mode()
def compute_all(text):
    torch.cuda.empty_cache()
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN, return_token_type_ids=False)
    eg = {k: v.to(DEVICE) for k, v in enc.items()}
    ol = observer(**eg).logits.float()
    pl = performer(**eg).logits.float()
    class E:
        def __init__(s, d): s.input_ids = d['input_ids']; s.attention_mask = d['attention_mask']
        def to(s, dev): s.input_ids = s.input_ids.to(dev); s.attention_mask = s.attention_mask.to(dev); return s
    eo = E(eg)
    ppl = sum_perplexity(eo, pl)
    xppl = entropy(ol, pl, eo, tokenizer.pad_token_id)
    rs = float(ppl[0] / (2 * xppl[0]))
    
    shifted = pl[..., :-1, :]; labels = eg['input_ids'][..., 1:]; attn = eg['attention_mask'][..., 1:]
    ce = F.cross_entropy(shifted.transpose(1,2), labels, reduction='none')
    S = ce[attn.bool()].cpu().numpy()
    if len(S) < 5: del ol, pl, eg; return None
    
    ce_var = float(np.var(S))
    
    obs_top = ol[..., :-1, :].argmax(dim=-1)
    perf_top = shifted.argmax(dim=-1)
    agree = (obs_top == perf_top).float()[attn.bool()].cpu().numpy()
    agree_rate = float(np.mean(agree))
    
    pe = -(F.softmax(pl[..., :-1, :], dim=-1) * F.log_softmax(pl[..., :-1, :], dim=-1)).sum(-1)
    H = pe[attn.bool()].cpu().numpy()
    H_med = np.median(H)
    conf = H < H_med
    coherence = float(np.mean(S[conf]) - np.mean(S[~conf])) if conf.sum() > 0 and (~conf).sum() > 0 else 0.0
    
    del ol, pl, eg
    return {'rs': rs, 'ce_var': ce_var, 'agree_rate': agree_rate, 'coherence': coherence}

print(f'Test: {compute_all("Hello world test sentence here.")}')

Test: {'rs': 0.9749189615249634, 'ce_var': 6.780651092529297, 'agree_rate': 0.800000011920929, 'coherence': -0.7117934226989746}


In [5]:
# Score all benchmarks
CKPT = 'rfinal_bench_ckpt.json'
if os.path.exists(CKPT):
    all_res = json.load(open(CKPT)); print(f'Checkpoint: {list(all_res.keys())}')
else:
    all_res = {}

for bname, bdata in dataset['benchmarks'].items():
    if bname in all_res: print(f'{bname}: done'); continue
    print(f'\nScoring {bname}...')
    h_sc = []; m_sc = []
    for t in tqdm(bdata['human'][:200], desc=f'{bname} human'):
        try: h_sc.append(compute_all(t))
        except: h_sc.append(None)
    for t in tqdm(bdata['machine'][:200], desc=f'{bname} machine'):
        try: m_sc.append(compute_all(t))
        except: m_sc.append(None)
    all_res[bname] = {'human': h_sc, 'machine': m_sc}
    with open(CKPT, 'w') as f: json.dump(all_res, f)
print('\nAll benchmarks scored!')


Scoring M4...


M4 machine: 100%|██████████| 200/200 [14:02<00:00,  4.21s/it]



Scoring DetectRL...


DetectRL machine: 100%|██████████| 200/200 [13:38<00:00,  4.09s/it]



Scoring RealDet...


RealDet machine: 100%|██████████| 200/200 [11:14<00:00,  3.37s/it]



Scoring XSum_GPT4...


XSum_GPT4 machine: 100%|██████████| 200/200 [14:29<00:00,  4.35s/it]


All benchmarks scored!


In [6]:
# Compute R_final and compare
# First get human baseline stats from pooled benchmarks
all_h_cev = []; all_h_agr = []; all_h_coh = []
for bname in all_res:
    for s in all_res[bname]['human']:
        if s:
            all_h_cev.append(s['ce_var'])
            all_h_agr.append(s['agree_rate'])
            all_h_coh.append(s['coherence'])
CEV_M, CEV_S = np.mean(all_h_cev), np.std(all_h_cev)
AGR_M, AGR_S = np.mean(all_h_agr), np.std(all_h_agr)
COH_M, COH_S = np.mean(all_h_coh), np.std(all_h_coh)
print(f'Pooled human baseline: ce_var={CEV_M:.2f}+/-{CEV_S:.2f}, agree={AGR_M:.4f}+/-{AGR_S:.4f}, coh={COH_M:.2f}+/-{COH_S:.2f}')

def R_final(s, k=1.0):
    rs = s['rs']
    p1 = max(0.1, 1 - 0.1 * max(0, s['ce_var'] - (CEV_M + k * CEV_S)))
    p2 = max(0.1, 1 - 3.0 * max(0, (AGR_M - k * AGR_S) - s['agree_rate']))
    p3 = max(0.1, 1 - 0.5 * max(0, (COH_M - k * COH_S) - s['coherence']))
    return rs * p1 * p2 * p3

print(f'\n{"Benchmark":<15} {"R(s) AUROC":>12} {"R_final AUROC":>14} {"Delta":>8}')
print('='*55)
output = {}
for bname in all_res:
    hv_r = [s['rs'] for s in all_res[bname]['human'] if s]
    mv_r = [s['rs'] for s in all_res[bname]['machine'] if s]
    hv_f = [R_final(s) for s in all_res[bname]['human'] if s]
    mv_f = [R_final(s) for s in all_res[bname]['machine'] if s]
    if not hv_r or not mv_r: continue
    auroc_r = roc_auc_score([1]*len(hv_r)+[0]*len(mv_r), hv_r+mv_r)
    auroc_f = roc_auc_score([1]*len(hv_f)+[0]*len(mv_f), hv_f+mv_f)
    delta = auroc_f - auroc_r
    print(f'{bname:<15} {auroc_r:>12.4f} {auroc_f:>14.4f} {delta:>+8.4f}')
    output[bname] = {'rs_auroc': auroc_r, 'rfinal_auroc': auroc_f, 'delta': delta,
                      'n_human': len(hv_r), 'n_machine': len(mv_r)}

print('\nIf Delta near 0: R_final preserves standard detection (GOOD)')
print('If Delta negative: R_final hurts standard detection (BAD)')

Pooled human baseline: ce_var=6.66+/-1.68, agree=0.7394+/-0.0588, coh=-2.44+/-0.52

Benchmark         R(s) AUROC  R_final AUROC    Delta
M4                    0.8923         0.8276  -0.0647
DetectRL              0.8807         0.9321  +0.0514
RealDet               0.9138         0.8232  -0.0906
XSum_GPT4             0.9948         0.9739  -0.0209

If Delta near 0: R_final preserves standard detection (GOOD)
If Delta negative: R_final hurts standard detection (BAD)


In [7]:
# Save
result = {'human_baseline': {'cev_m': CEV_M, 'cev_s': CEV_S, 'agr_m': AGR_M, 'agr_s': AGR_S, 'coh_m': COH_M, 'coh_s': COH_S}, 'benchmarks': output}
with open('rfinal_benchmark_results.json', 'w') as f: json.dump(result, f, indent=2)
print('Saved rfinal_benchmark_results.json')

Saved rfinal_benchmark_results.json
